# Episode 4: Make Messy Data Feel Manageable (Flower Gift Sales)
**Data Science with Isa · Hire-Ready Data Science**

**Business question:** Which weeks and weekdays bring this shop the most flower-gift revenue?

**Data:** "Online Retail" by Daqing Chen, UCI Machine Learning Repository (2015), CC BY 4.0: https://archive.ics.uci.edu/dataset/352/online+retail
I kept only the rows whose product description mentions FLOWER, ROSE, VASE, GARDEN or PLANT (28,842 rows). Nothing else was changed. The messy parts are left in on purpose, so we can practise cleaning.

The copy we load is pinned (tag `ep4-data-v1` in my GitHub repo). That means everyone loads exactly the same file, with no login, and gets exactly the same numbers as in the video.

**What you'll practise:** inspecting a new dataset, fixing data types, `to_datetime`, deciding about missing values, removing duplicates, separating cancelled orders, `.dt.day_name()`, `groupby`, `resample` to weekly totals, a line chart, and writing an answer with honest limits.

Run the cells from top to bottom. Each code cell has an explanation above it: what it does, what each part means, what you should see, and why it matters.

## 0. Setup

**What this cell does:** it loads the tools (libraries) we need and prints their versions.

**What each part means:**
- `import sys` gives us information about Python itself. We only use it to print the Python version.
- `import pandas as pd` loads **pandas**, the library for tables. A pandas table is called a **DataFrame**. `as pd` is a short nickname, so we can type `pd` instead of `pandas`.
- `import matplotlib` and `import matplotlib.pyplot as plt` load **matplotlib**, the library for charts. `plt` is its usual nickname.
- `print(...)` writes text under the cell. `sys.version.split()[0]` keeps only the version number, like `3.13.5`. `pd.__version__` and `matplotlib.__version__` are the library versions.

**What you should see:** `Python 3.13.5 | pandas 3.0.6 | matplotlib 3.11.2`

**Why it matters:** different versions can show results a little differently. To match my numbers exactly, install the same versions. Run this in a cell, then restart the kernel:

`%pip install pandas==3.0.6 matplotlib==3.11.2`

With an older pandas, text columns show as `object` instead of `str`. It means the same thing.

Tip: if a cell shows `[*]` and nothing happens, the kernel is busy or stuck. Wait a bit, or use **Kernel → Interrupt**, then **Restart**.

In [ ]:
import sys
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| matplotlib", matplotlib.__version__)

## 1. Load and inspect

**What this cell does:** it downloads the data from my GitHub and loads it into a table called `df`. Then it shows the size of the table.

**What each part means:**
- `url = "..."` saves the web address of the CSV file in a variable called `url`. A **CSV** file is a plain text table: one row per line, with commas between the values.
- `ep4-data-v1` inside the address is a fixed version (a **tag**). It never changes, so the file never changes.
- `pd.read_csv(url)` reads the CSV file and turns it into a DataFrame. It works with a web address or a file on your computer.
- `df` is the usual short name for "data frame".
- `df.shape` gives the size as **(rows, columns)**.

**What you should see:** `(28842, 8)`, so 28,842 rows and 8 columns. Each row is one product on one invoice.

**Why it matters:** checking the size is the first test that you loaded the right file. If you don't see `(28842, 8)`, stop and check the link. You need internet for this cell.

In [ ]:
url = "https://raw.githubusercontent.com/isajafari1989/Data-Science-With-Isa/ep4-data-v1/data/flower_gifts_sales_raw.csv"
df = pd.read_csv(url)
df.shape

**What this cell does:** it shows the first 5 rows of the table.

**What each part means:**
- `df.head()` shows the top 5 rows. `df.head(10)` would show 10.
- The 8 columns are:
  - `InvoiceNo`: the invoice (order) number. If it starts with **C**, the order was cancelled.
  - `StockCode`: the product code.
  - `Description`: the product name, for example `ENAMEL FLOWER JUG CREAM`.
  - `Quantity`: how many items were bought.
  - `InvoiceDate`: the date and time of the order.
  - `UnitPrice`: the price of one item, in British pounds (GBP).
  - `CustomerID`: the customer's number. Sometimes it's empty.
  - `Country`: where the customer lives.

**What you should see:** the first row is invoice `536381`, `SMALL HEART FLOWERS HOOK`, quantity `6`, on `2010-12-01 09:41:00`, price `0.85`.

**Why it matters:** looking at real rows helps you understand what one row means before you count or add anything. Notice that `CustomerID` shows `15311.0` with a `.0`. That's a hint that the column type isn't ideal. We'll fix it later.

In [ ]:
df.head()

### Check 1: data types

**What this cell does:** it shows the **data type** (dtype, say "D-type") of every column.

**What each part means:**
- `df.dtypes` lists each column with its type:
  - `str`: text (older pandas shows `object`)
  - `int64`: whole numbers
  - `float64`: decimal numbers
  - `datetime64`: real dates and times

**What you should see:** `Quantity` is `int64`, `UnitPrice` and `CustomerID` are `float64`, and `InvoiceDate` is `str`.

**Why it matters:** `InvoiceDate` is stored as **text**, not as a date. With text, pandas can't tell you the weekday or group by week. This is a common problem with CSV files, because a CSV file doesn't remember types. `CustomerID` is a decimal number only because some values are missing (pandas uses decimals when a number column has empty cells).

In [ ]:
df.dtypes

### Check 2: missing values

**What this cell does:** it counts the empty (missing) cells in each column.

**What each part means:**
- `df.isna()` checks every cell and gives `True` if it's empty and `False` if it's not. `isna` means "is not available". Empty cells show as `NaN` ("not a number").
- `.sum()` adds them up for each column. `True` counts as 1 and `False` as 0.

**What you should see:** `0` for every column, except `CustomerID` with **8071**.

**Why it matters:** missing values can quietly change your answer. Before you decide what to do with them, you need to know where they are and how many there are. We'll make the decision in step 3.

In [ ]:
df.isna().sum()

### Check 3: duplicates

**What this cell does:** it counts the rows that are exact copies of an earlier row.

**What each part means:**
- `df.duplicated()` gives `True` for a row that matches an earlier row in **every** column (same invoice, product, quantity, date and minute, price, customer and country).
- `.sum()` counts the `True` values.

**What you should see:** `np.int64(265)`, so **265** duplicate rows. (`np.int64` just means "a whole number". You can read it as `265`.)

**Why it matters:** if the same sale appears twice, we count its money twice. That makes revenue too high.

In [ ]:
df.duplicated().sum()

### Check 4: strange values

**What this cell does:** it counts the rows with a negative quantity, and checks if their invoice number starts with "C".

**What each part means:**
- `df["Quantity"] < 0` checks every row: `True` if the quantity is below zero.
- `(...).sum()` counts them.
- `df.loc[rows, "InvoiceNo"]` picks the `InvoiceNo` values only for the chosen rows. `loc` means "locate": rows first, then the column.
- `.str.startswith("C")` checks if the text starts with the letter C. `.str` lets us use text tools on a whole column.
- `.mean()` of `True`/`False` values gives the share that are `True`. `1.0` means 100%.
- The comma makes Python show both answers together, in brackets.

**What you should see:** `(np.int64(533), np.float64(1.0))`, so **533** rows have a negative quantity, and **all** of them (1.0 = 100%) have an invoice starting with **C**.

**Why it matters:** a negative quantity is a cancelled order or a return, not a sale. If we add them to sales, revenue goes down by mistake. Now we know the rule: invoices starting with C are cancellations.

In [ ]:
# rows with a negative quantity, and how many of them have an invoice starting with "C"
(df["Quantity"] < 0).sum(), df.loc[df["Quantity"] < 0, "InvoiceNo"].str.startswith("C").mean()

## 2. Fix data types

**What this cell does:** it turns the date column from text into real dates, makes sure the invoice number is text, and shows the types again.

**What each part means:**
- `pd.to_datetime(df["InvoiceDate"])` reads text like `2010-12-01 09:41:00` and turns it into a real date and time.
- `df["InvoiceDate"] = ...` saves the result back into the same column.
- `.astype(str)` changes a column's type to text. Invoice numbers are labels, not amounts. We never add them up, and some have a letter C in them.
- `df.dtypes` shows the types again, so we can check that it worked.

**What you should see:** `InvoiceDate` is now `datetime64[us]`. (`[us]` means it keeps time down to microseconds. With older pandas you may see `[ns]`, nanoseconds. Both are fine.)

**Why it matters:** now pandas understands the dates. It knows the weekday, the week and the month of every order, and it can sort them in time order.

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df.dtypes

**What this cell does:** it shows the first and the last date in the data.

**What each part means:**
- `.min()` gives the smallest value. For dates, that's the earliest.
- `.max()` gives the largest value. For dates, that's the latest.
- `Timestamp(...)` is how pandas shows one date and time.

**What you should see:** `2010-12-01 09:41:00` to `2011-12-09 11:56:00`. That's about one year of sales.

**Why it matters:** it's a quick test that the dates were read correctly (no strange years, and day and month not swapped). It also tells us the time range of our answer.

In [ ]:
# now the dates are real dates, so we can ask for the first and last one
df["InvoiceDate"].min(), df["InvoiceDate"].max()

## 3. Missing values: drop or fill?

Only **CustomerID** has missing values (8,071 rows). We have two common choices:
- **Fill:** put a value in the empty cells. That's wrong here. An invented customer number would be a person who doesn't exist.
- **Drop:** delete the rows. That's also wrong for our question. A sale without a customer number is still a sale, and the money still came in.

**Decision:** keep these rows for revenue questions. Drop them only for questions about customers. So there's no code to run here. We'll measure how much money these rows hold in step 5.

## 4. Duplicates

**What this cell does:** it removes the exact duplicate rows and counts the rows left.

**What each part means:**
- `df.drop_duplicates()` keeps the first copy of each row and removes the extra copies.
- `df = ...` saves the cleaned table under the same name. Without this, the change would be shown but not kept.
- `len(df)` gives the number of rows.

**What you should see:** **28577**. That's 28,842 − 265 = 28,577, so exactly the 265 duplicates were removed.

**Why it matters:** each sale is now counted once. And checking the row count proves the step did what we expected.

In [ ]:
df = df.drop_duplicates()
len(df)

## 5. Cancelled orders and odd rows

**What this cell does:** it splits the data into two tables: `returns` (cancelled orders) and `sales` (real sales).

**What each part means:**
- `df[condition]` keeps only the rows where the condition is `True`. This is called **filtering**.
- `df["InvoiceNo"].str.startswith("C")` is `True` for cancelled invoices. So `returns` is all the C-invoices.
- `~` means "NOT". So `~df["InvoiceNo"].str.startswith("C")` means "the invoice does **not** start with C".
- `&` means "AND": all the conditions must be true. Each condition needs its own brackets.
- `(df["Quantity"] > 0)` keeps rows where something was bought. `(df["UnitPrice"] > 0)` keeps rows that cost money.
- `.copy()` makes `sales` its own separate table, so we can add columns to it safely.
- `len(returns), len(sales)` counts the rows in each table.

**What you should see:** `(529, 28013)`, so **529** cancellations and **28,013** sales rows.

**Why it matters:** cancellations are real events, but they're not sales. We keep them in their own table (we could study them later) instead of deleting them, and we keep them out of revenue.

In [ ]:
returns = df[df["InvoiceNo"].str.startswith("C")]
sales = df[~df["InvoiceNo"].str.startswith("C") & (df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
len(returns), len(sales)

**What this cell does:** it finds the rows that aren't cancellations but have a price of zero.

**What each part means:**
- `df["UnitPrice"] == 0` is `True` when the price is exactly zero. `==` means "is equal to" (one `=` saves a value, two `==` compare).
- `~...startswith("C")` again means "not a cancellation".
- `free` is the name for this small table, and `len(free)` counts its rows.

**What you should see:** **35**.

**Why it matters:** our numbers must add up. 28,577 rows = 529 returns + 28,013 sales + **35** zero-price rows. These 35 rows brought no money, so they don't belong in revenue. Always check that your counts add up. If they don't, something is hiding.

In [ ]:
# rows that are not cancellations but have a price of zero
free = df[~df["InvoiceNo"].str.startswith("C") & (df["UnitPrice"] == 0)]
len(free)

**What this cell does:** it fixes the `CustomerID` type, makes a new `Revenue` column, and adds up all the revenue.

**What each part means:**
- `.astype("Int64")` (with a capital **I**) turns `CustomerID` into whole numbers that **can** have empty values. So `15311.0` becomes `15311`, and the empty cells stay empty (shown as `<NA>`). The normal `int64` type can't hold empty cells.
- `sales["Quantity"] * sales["UnitPrice"]` multiplies the two columns, row by row, all at once (no loop needed). For example, 6 items × 0.85 = 5.10.
- `sales["Revenue"] = ...` saves the result as a new column called `Revenue`.
- `.sum()` adds up the whole column, and `.round(2)` keeps 2 decimals.

**What you should see:** `np.float64(467418.75)`, so total flower-gift revenue is **£467,418.75**. (`np.float64` just means "a decimal number".)

**Why it matters:** our business question is about revenue, so we need one clear revenue number for every row.

In [ ]:
sales["CustomerID"] = sales["CustomerID"].astype("Int64")
sales["Revenue"] = sales["Quantity"] * sales["UnitPrice"]
sales["Revenue"].sum().round(2)

**What this cell does:** it goes back to our missing values decision and measures how important those rows are.

**What each part means:**
- `sales["CustomerID"].isna()` is `True` for sales rows with no customer number. We save it as `no_id`.
- `no_id.mean()` gives the share of rows with no customer (the share of `True` values).
- `sales.loc[no_id, "Revenue"].sum()` adds up the revenue of only those rows. Dividing by the total revenue gives their share of the money.
- `.round(3)` keeps 3 decimals.

**What you should see:** `(0.286, 0.151)`. About **28.6%** of sales rows have no customer number, and they bring about **15.1%** of all revenue.

**Why it matters:** this proves the decision was right. If we had dropped these rows, we would have lost about 15% of the money, and every weekday and week total would be too low.

In [ ]:
# back to missing values: share of sales rows with no CustomerID, and their share of revenue
no_id = sales["CustomerID"].isna()
no_id.mean().round(3), (sales.loc[no_id, "Revenue"].sum() / sales["Revenue"].sum()).round(3)

## 6. Dates: weekdays and weeks

**What this cell does:** it adds a weekday name to every sale, then adds up revenue for each weekday, from highest to lowest.

**What each part means:**
- `.dt` opens the date tools for a date column. It only works because we used `to_datetime` in step 2.
- `.dt.day_name()` gives the weekday name, like `Thursday`. We save it as a new column, `Weekday`.
- `sales.groupby("Weekday")` splits the rows into groups, one group per weekday.
- `["Revenue"].sum()` adds up the revenue in each group. So: split, calculate, combine.
- `.sort_values(ascending=False)` sorts from biggest to smallest. (`ascending=True` would be smallest first.)
- `.round(2)` shows 2 decimals.

**What you should see:**

| Weekday | Revenue (GBP) |
|---|---|
| Thursday | 114,916.12 |
| Tuesday | 84,812.19 |
| Wednesday | 80,440.08 |
| Monday | 74,624.43 |
| Friday | 73,011.24 |
| Sunday | 39,614.69 |

There's **no Saturday** at all. This shop didn't sell on Saturdays.

**Why it matters:** this is the first half of our answer: which weekdays bring the most revenue.

In [ ]:
sales["Weekday"] = sales["InvoiceDate"].dt.day_name()
by_weekday = sales.groupby("Weekday")["Revenue"].sum().sort_values(ascending=False)
by_weekday.round(2)

**What this cell does:** it turns each weekday's revenue into a percentage of the total.

**What each part means:**
- `by_weekday / by_weekday.sum()` divides each day's revenue by the total of all days. This gives a share between 0 and 1.
- `* 100` turns it into a percentage.
- `.round(1)` keeps 1 decimal.

**What you should see:** Thursday **24.6**, Tuesday 18.1, Wednesday 17.2, Monday 16.0, Friday 15.6, Sunday **8.5**.

**Why it matters:** percentages are easier to explain than big numbers. "Thursday brings about a quarter of the revenue" is a sentence anyone can understand.

In [ ]:
# each weekday's share of revenue, in percent
(by_weekday / by_weekday.sum() * 100).round(1)

**What this cell does:** it adds up revenue for every week, then counts the weeks and finds the average week.

**What each part means:**
- `sales.set_index("InvoiceDate")` makes the date column the **index** (the row labels on the left). `resample` needs dates there.
- `["Revenue"]` picks the revenue column.
- `.resample("W")` groups the rows into time periods. `"W"` means weekly. Each week runs Monday to Sunday and is labelled with its **Sunday** date. (`"D"` would be daily.)
- `.sum()` adds up the revenue in each week. We save the result as `weekly`.
- `len(weekly)` counts the weeks, and `weekly.mean()` gives the average revenue per week.

**What you should see:** `(54, np.float64(8655.9))`, so **54** weeks and an average of about **£8,655.90** per week.

**Why it matters:** with one line, thousands of rows become one number per week. That's how we see patterns over time.

In [ ]:
weekly = sales.set_index("InvoiceDate")["Revenue"].resample("W").sum()
len(weekly), weekly.mean().round(2)

**What this cell does:** it shows the 5 weeks with the most revenue.

**What each part means:**
- `weekly.sort_values(ascending=False)` sorts the weeks from the most revenue to the least.
- `.head()` keeps the top 5.
- Each date is the **Sunday that ends** that week.

**What you should see:**
- week ending 2011-05-22: **15,113.80**
- week ending 2011-10-23: 14,729.44
- week ending 2011-07-03: 13,167.16
- week ending 2011-06-12: 12,282.96
- week ending 2011-02-27: 12,176.76

**Why it matters:** this is the second half of our answer: which weeks bring the most revenue. The best week ended on 22 May 2011.

In [ ]:
# top five weeks (each label is the Sunday that ends the week)
weekly.sort_values(ascending=False).head().round(2)

**What this cell does:** it shows the 3 weeks with the least revenue.

**What each part means:**
- `weekly.sort_values()` sorts from smallest to largest (that's the default).
- `.head(3)` keeps the first 3.

**What you should see:**
- week ending 2011-01-02: **0.00**
- week ending 2010-12-26: 3,159.55
- week ending 2011-05-01: 4,252.52

**Why it matters:** a zero week looks like an error, but it isn't. There were simply no sales over the new year holidays. Always look at the lowest values too. Messy data often has a real story behind it.

In [ ]:
# quietest three weeks
weekly.sort_values().head(3).round(2)

**What this cell does:** it draws a line chart of weekly revenue and saves it as a picture.

**What each part means:**
- `plt.rcParams.update({"font.size": 14})` makes all the chart text bigger and easier to read.
- `weekly.plot(...)` draws a line chart. `figsize=(12, 5)` is the size (width, height). `color="#3733a2"` is the line colour, written as a colour code.
- `ax` is the chart area. `ax.set(...)` adds the title and the axis labels.
- `ax.spines[["top", "right"]].set_visible(False)` hides the top and right border lines, for a cleaner look.
- `plt.tight_layout()` stops the labels getting cut off. `plt.savefig("weekly_revenue.png", dpi=150)` saves the chart as a PNG picture. `dpi` is the sharpness.
- The `;` lets us put two commands on one line.

**What you should see:** a line that moves around about £8,700 a week, with high points in **late May** and **October**, and a drop to **zero** at the new year.

**Why it matters:** a chart shows the pattern at a glance. Keep it plain, so the data is the star.

In [ ]:
plt.rcParams.update({"font.size": 14})
ax = weekly.plot(figsize=(12, 5), color="#3733a2")
ax.set(title="Weekly flower-gift revenue (GBP)", xlabel="Week", ylabel="Revenue")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig("weekly_revenue.png", dpi=150)

## 7. Answer and limits

**Answer:** Thursday brings the most flower-gift revenue: about £114,916, or 24.6% of the total. Tuesday is second with 18.1%, and Sunday is the quietest day with 8.5%. The best week was the week ending 22 May 2011, with about £15,114. An average week brings about £8,656. A next step: check what sold in that best week.

**Limits:**
- This is one UK online gift shop, from December 2010 to December 2011, so about one year.
- It's flower gifts only, picked by keywords, not the whole shop. The keyword filter isn't perfect: for example, ROSE also matches ROSETTE.
- There are no Saturday sales in this data, so we can't say anything about Saturdays.
- The first and last weeks are partial weeks, and the week ending 2 January 2011 has no sales at all.
- A few very large orders can push one week up.

## 8. Save

**What this cell does:** it saves the clean sales table as a new CSV file.

**What each part means:**
- `sales.to_csv("flower_gifts_sales_clean.csv")` writes the `sales` table to a file in the same folder as this notebook.
- `index=False` means "don't save the row numbers as an extra column".

**What you should see:** no output under the cell. A new file, `flower_gifts_sales_clean.csv`, appears in your folder.

**Why it matters:** the raw file stays untouched, and the clean file is ready for the next analysis. Next, save this notebook and commit it to Git:

`git add 04-messy-data.ipynb`
`git commit -m "Clean flower gift sales and find best weekdays"`
`git push`

In [ ]:
sales.to_csv("flower_gifts_sales_clean.csv", index=False)